# 04 — Worker Paralel 1: Semua Eksperimen Image-Only (5-Fold CV)

Notebook ini khusus menjalankan **15 eksperimen unik Image-Only** secara paralel dengan notebook `03` (Multimodal Worker 2), tanpa duplikasi maupun bentrokan folder di `runs/`.

**Daftar Eksperimen (Total 15 run x 5 fold):**
1. **Bagian 1 — B2 Aggregator Image-Only (9 run):**
   - `task_mode`: `binary_diag`, `binary_sev`, `hierarchical`
   - `aggregator`: `mv_mean`, `mv_lstm`, `cross_attn`
   - Disimpan ke: `runs/02_b2_image_only_{task_mode}_{agg}` (bag `all`).
2. **Bagian 2 — Bag/View Ablation Image-Only (6 run baru + 3 reuse):**
   - `task_mode`: `hierarchical`, `binary_diag`, `binary_sev`
   - `bag_mode="all"` **di-reuse otomatis** dari `runs/02_b2_image_only_{task_mode}_cross_attn` (0 training ulang!).
   - `bag_mode` `appendix_single` dan `appendix_plus_1_2` dilatih dan disimpan ke: `runs/03_bag_image_only_{task_mode}_{bmode}`.

> **Catatan:** Pastikan `BEST_BACKBONE` disamakan dengan pemenang `02` Bagian 1 sebelum menjalankan notebook ini. Setiap proses memakai ~0,98 GiB VRAM sehingga aman dijalankan bersamaan dengan `03` di GPU RTX 3060 6GB.


In [1]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PM3_DIR = Path.cwd()
if not (PM3_DIR / "pm3_data.py").exists():
    PM3_DIR = PM3_DIR / "Proposed Model 3"
assert (PM3_DIR / "pm3_data.py").exists(), f"Tidak menemukan pm3_data.py di {PM3_DIR}"
if str(PM3_DIR) not in sys.path:
    sys.path.insert(0, str(PM3_DIR))

from pm3_data import USConfig, build_image_cache, images_by_patient, load_cohort, make_stratified_folds
from pm3_model import ModelConfig
from pm3_train import TrainConfig, run_experiment

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM free/total:",
          [round(x / 1024**3, 2) for x in torch.cuda.mem_get_info()])
else:
    print("Device:", DEVICE)

ROOT_DIR = PM3_DIR.parent
cohort = load_cohort(ROOT_DIR / "app_data_filtered.csv")
manifest = pd.read_csv(ROOT_DIR / "image_manifest_filtered.csv")
folds = make_stratified_folds(cohort, seed=1, n_splits=5)
cache = build_image_cache(manifest, ROOT_DIR / "US_Pictures_clean", USConfig(), PM3_DIR / "cache")

bags_by_mode = {
    bmode: images_by_patient(manifest, bag_mode=bmode)
    for bmode in ("all", "appendix_single", "appendix_plus_1_2")
}
print(f"Cohort: {len(cohort)} pasien | Folds: {len(folds)} | Cache entries: {len(cache)}")

def row_from_res(meta: dict, res: dict, run_dir: Path) -> dict:
    cal = res.get("oof_calibrated") or res.get("oof_fixed") or {}
    row = dict(meta)
    row["run_dir"] = run_dir.name
    for k, v in cal.items():
        if isinstance(v, (int, float, np.floating, np.integer)):
            row[k] = float(v)
    return row

def try_load_summary(run_dir: Path):
    summary_path = run_dir / "summary_metrics.json"
    if not summary_path.exists():
        return None
    try:
        return json.loads(summary_path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"[WARN] Gagal membaca {summary_path}: {exc}")
        return None


GPU: NVIDIA GeForce RTX 3060 Laptop GPU | VRAM free/total: [5.0, 6.0]
Preprocessing/cache PM3: 250/1747
Preprocessing/cache PM3: 500/1747
Preprocessing/cache PM3: 750/1747
Preprocessing/cache PM3: 1000/1747
Preprocessing/cache PM3: 1250/1747
Preprocessing/cache PM3: 1500/1747
Cohort: 578 pasien | Folds: 5 | Cache entries: 1747


In [2]:
# ==================== KONFIGURASI WORKER 1 (IMAGE-ONLY) ====================
BEST_BACKBONE = "stage7_only"  # <-- Sesuaikan dengan pemenang 02 Bagian 1!

RUN_B2_IMAGE_ONLY = True
RUN_BAG_IMAGE_ONLY = True
SKIP_IF_DONE = True  # True: otomatis skip fold/eksperimen yang sudah selesai

TASKS = ("binary_diag", "binary_sev", "hierarchical")
AGGS = ("mv_mean", "mv_lstm", "cross_attn")
BAG_MODES = ("appendix_single", "appendix_plus_1_2", "all")

def make_train_cfg(bag_mode: str = "all") -> TrainConfig:
    return TrainConfig(
        epochs=60,
        batch_size=16,
        patience=15,
        warmup_freeze_epochs=5,
        seed=1,
        bag_mode=bag_mode,
    )

print(f"Worker 1 Image-Only siap | BEST_BACKBONE={BEST_BACKBONE} | SKIP_IF_DONE={SKIP_IF_DONE}")


Worker 1 Image-Only siap | BEST_BACKBONE=stage7_only | SKIP_IF_DONE=True


## Bagian 1 — B2 Aggregator Ablation (Image-Only, 9 Run Unik)
Menjalankan kombinasi `TASKS x AGGS` pada mode `image_only` dengan `bag_mode="all"`.


In [3]:
b2_img_rows = []
if RUN_B2_IMAGE_ONLY:
    for task_mode in TASKS:
        for agg in AGGS:
            run_dir = PM3_DIR / "runs" / f"02_b2_image_only_{task_mode}_{agg}"
            label = f"04_b2_img/{task_mode}/{agg}"
            print(f"\n>>> [{label}] -> {run_dir.name}")
            mcfg = ModelConfig(
                clinical_dim=28,
                num_concepts=8,
                modality="image_only",
                backbone_mode=BEST_BACKBONE,
                aggregator=agg,
                fusion_mode="film",
                task_mode=task_mode,
                use_concepts=True,
            )
            tcfg = make_train_cfg("all")
            res = run_experiment(
                cohort=cohort,
                folds=folds,
                bags=bags_by_mode["all"],
                cache=cache,
                model_cfg=mcfg,
                train_cfg=tcfg,
                directory=run_dir,
                device=DEVICE,
                verbose=True,
                skip_if_done=SKIP_IF_DONE,
                variant_label=label,
            )
            b2_img_rows.append(
                row_from_res(
                    {"modality": "image_only", "task_mode": task_mode, "aggregator": agg, "backbone_mode": BEST_BACKBONE, "bag_mode": "all"},
                    res,
                    run_dir,
                )
            )
    df_b2_img = pd.DataFrame(b2_img_rows)
    out_csv = PM3_DIR / "runs" / "04_image_only_b2_summary.csv"
    df_b2_img.to_csv(out_csv, index=False)
    print(f"\nTersimpan ringkasan B2 Image-Only -> {out_csv}")
    display(df_b2_img)
else:
    print("RUN_B2_IMAGE_ONLY=False, dilewati.")



>>> [04_b2_img/binary_diag/mv_mean] -> 02_b2_image_only_binary_diag_mv_mean

04_b2_img/binary_diag/mv_mean | fold 1 | epoch 1/60 | best epoch 1 | backbone=FROZEN(warmup) | 42.5s
TRAIN | Loss 0.770819 | Acc 0.5195 | F1 macro 0.5107 | AUROC 0.5578
  loss raw      : binary=0.469743 | concept=0.859931 | aux_img=0.000000
  loss weighted : binary=0.469743 | concept=0.300976 | aux_img=0.000000
VAL   | Loss 0.763116 | Acc 0.6724 | Prec 0.7262 | Sens 0.8026 | Spec 0.4250 | F1 pos 0.7625 | F1 macro 0.6174 | AUROC 0.6387 | AUPR 0.7324 | Concept AUC 0.5174 | Concept F1 0.4356
  loss raw      : binary=0.459478 | concept=0.865937 | aux_img=0.000000
  loss weighted : binary=0.459478 | concept=0.303078 | aux_img=0.000000
Confusion matrix VAL (baris=label asli, kolom=prediksi):
             0: Negative  1: Positive
0: Negative           17           23
1: Positive           15           61
Kesalahan VAL per kelas asli (rata-rata probabilitas kasus salah):
 kelas_asli  n_total  n_salah  p_0_mean  p_1_m

,modality,task_mode,aggregator,backbone_mode,bag_mode,run_dir,accuracy,precision,sensitivity,specificity,...,specificity_class_1,f1_class_1,auroc_class_1,aupr_class_1,precision_class_2,sensitivity_class_2,specificity_class_2,f1_class_2,auroc_class_2,aupr_class_2
0,image_only,binary_diag,mv_mean,stage7_only,all,02_b2_image_only_binary_diag_mv_mean,0.681661,0.769022,0.740838,0.566327,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,image_only,binary_diag,mv_lstm,stage7_only,all,02_b2_image_only_binary_diag_mv_lstm,0.764706,0.804455,0.850785,0.596939,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,image_only,binary_diag,cross_attn,stage7_only,all,02_b2_image_only_binary_diag_cross_attn,0.664360,0.759669,0.719895,0.556122,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,image_only,binary_sev,mv_mean,stage7_only,all,02_b2_image_only_binary_sev_mv_mean,0.807958,0.438017,0.552083,0.858921,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,image_only,binary_sev,mv_lstm,stage7_only,all,02_b2_image_only_binary_sev_mv_lstm,0.757785,0.325397,0.427083,0.823651,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,image_only,binary_sev,cross_attn,stage7_only,all,02_b2_image_only_binary_sev_cross_attn,0.811419,0.443478,0.531250,0.867220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,image_only,hierarchical,mv_mean,stage7_only,all,02_b2_image_only_hierarchical_mv_mean,0.527682,NaN,NaN,NaN,...,0.595890,0.584939,0.616175,0.612822,0.426230,0.541667,0.854772,0.477064,0.775026,0.435248
7,image_only,hierarchical,mv_lstm,stage7_only,all,02_b2_image_only_hierarchical_mv_lstm,0.634948,NaN,NaN,NaN,...,0.726027,0.690519,0.743426,0.747231,0.433333,0.541667,0.858921,0.481481,0.773989,0.403910
8,image_only,hierarchical,cross_attn,stage7_only,all,02_b2_image_only_hierarchical_cross_attn,0.500000,NaN,NaN,NaN,...,0.602740,0.551351,0.632256,0.608668,0.350877,0.416667,0.846473,0.380952,0.757715,0.400952


## Bagian 2 — Bag/View Ablation (Image-Only, 6 Run Baru + 3 Reuse)
Membandingkan `appendix_single`, `appendix_plus_1_2`, dan `all` (di-reuse dari `02_b2_image_only_{task_mode}_cross_attn`).


In [4]:
bag_img_rows = []
if RUN_BAG_IMAGE_ONLY:
    for task_mode in TASKS:
        for bmode in BAG_MODES:
            if bmode == "all":
                run_dir = PM3_DIR / "runs" / f"02_b2_image_only_{task_mode}_cross_attn"
                source_tag = "reused_from_b2"
            else:
                run_dir = PM3_DIR / "runs" / f"03_bag_image_only_{task_mode}_{bmode}"
                source_tag = "trained_in_04"

            label = f"04_bag_img/{task_mode}/{bmode}"
            print(f"\n>>> [{label}] ({source_tag}) -> {run_dir.name}")
            mcfg = ModelConfig(
                clinical_dim=28,
                num_concepts=8,
                modality="image_only",
                backbone_mode=BEST_BACKBONE,
                aggregator="cross_attn",
                fusion_mode="film",
                task_mode=task_mode,
                use_concepts=True,
            )
            tcfg = make_train_cfg(bmode)
            res = run_experiment(
                cohort=cohort,
                folds=folds,
                bags=bags_by_mode[bmode],
                cache=cache,
                model_cfg=mcfg,
                train_cfg=tcfg,
                directory=run_dir,
                device=DEVICE,
                verbose=True,
                skip_if_done=SKIP_IF_DONE,
                variant_label=label,
            )
            bag_img_rows.append(
                row_from_res(
                    {"modality": "image_only", "task_mode": task_mode, "aggregator": "cross_attn", "backbone_mode": BEST_BACKBONE, "bag_mode": bmode, "source": source_tag},
                    res,
                    run_dir,
                )
            )
    df_bag_img = pd.DataFrame(bag_img_rows)
    out_csv = PM3_DIR / "runs" / "04_image_only_bag_summary.csv"
    df_bag_img.to_csv(out_csv, index=False)
    print(f"\nTersimpan ringkasan Bag Image-Only -> {out_csv}")
    display(df_bag_img)
else:
    print("RUN_BAG_IMAGE_ONLY=False, dilewati.")



>>> [04_bag_img/binary_diag/appendix_single] (trained_in_04) -> 03_bag_image_only_binary_diag_appendix_single

04_bag_img/binary_diag/appendix_single | fold 1 | epoch 1/60 | best epoch 1 | backbone=FROZEN(warmup) | 10.5s
TRAIN | Loss 0.779736 | Acc 0.5152 | F1 macro 0.4994 | AUROC 0.5185
  loss raw      : binary=0.477145 | concept=0.864154 | aux_img=0.000000
  loss weighted : binary=0.477145 | concept=0.302454 | aux_img=0.000000
VAL   | Loss 0.746726 | Acc 0.5948 | Prec 0.7636 | Sens 0.5526 | Spec 0.6750 | F1 pos 0.6412 | F1 macro 0.5879 | AUROC 0.6734 | AUPR 0.7977 | Concept AUC 0.5140 | Concept F1 0.4638
  loss raw      : binary=0.440454 | concept=0.872984 | aux_img=0.000000
  loss weighted : binary=0.440454 | concept=0.305545 | aux_img=0.000000
Confusion matrix VAL (baris=label asli, kolom=prediksi):
             0: Negative  1: Positive
0: Negative           27           13
1: Positive           34           42
Kesalahan VAL per kelas asli (rata-rata probabilitas kasus salah):
 ke

,modality,task_mode,aggregator,backbone_mode,bag_mode,source,run_dir,accuracy,precision,sensitivity,...,specificity_class_1,f1_class_1,auroc_class_1,aupr_class_1,precision_class_2,sensitivity_class_2,specificity_class_2,f1_class_2,auroc_class_2,aupr_class_2
0,image_only,binary_diag,cross_attn,stage7_only,appendix_single,trained_in_04,03_bag_image_only_binary_diag_appendix_single,0.617647,0.729345,0.670157,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,image_only,binary_diag,cross_attn,stage7_only,appendix_plus_1_2,trained_in_04,03_bag_image_only_binary_diag_appendix_plus_1_2,0.610727,0.743034,0.628272,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,image_only,binary_diag,cross_attn,stage7_only,all,reused_from_b2,02_b2_image_only_binary_diag_cross_attn,0.664360,0.759669,0.719895,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,image_only,binary_sev,cross_attn,stage7_only,appendix_single,trained_in_04,03_bag_image_only_binary_sev_appendix_single,0.826990,0.480769,0.520833,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,image_only,binary_sev,cross_attn,stage7_only,appendix_plus_1_2,trained_in_04,03_bag_image_only_binary_sev_appendix_plus_1_2,0.792388,0.401639,0.510417,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,image_only,binary_sev,cross_attn,stage7_only,all,reused_from_b2,02_b2_image_only_binary_sev_cross_attn,0.811419,0.443478,0.531250,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,image_only,hierarchical,cross_attn,stage7_only,appendix_single,trained_in_04,03_bag_image_only_hierarchical_appendix_single,0.489619,NaN,NaN,...,0.650685,0.513410,0.596681,0.582402,0.385827,0.510417,0.838174,0.439462,0.784794,0.434718
7,image_only,hierarchical,cross_attn,stage7_only,appendix_plus_1_2,trained_in_04,03_bag_image_only_hierarchical_appendix_plus_1_2,0.538062,NaN,NaN,...,0.664384,0.564486,0.633083,0.609608,0.414634,0.531250,0.850622,0.465753,0.772930,0.439062
8,image_only,hierarchical,cross_attn,stage7_only,all,reused_from_b2,02_b2_image_only_hierarchical_cross_attn,0.500000,NaN,NaN,...,0.602740,0.551351,0.632256,0.608668,0.350877,0.416667,0.846473,0.380952,0.757715,0.400952
